# Negotiating Response Format for Multi-Agent Communication

**Objective:** Show how agents that speak different data formats can still talk to each other reliably. Each agent **declares** the formats it supports, the pair **negotiates** a shared format at runtime, the payload is **converted** automatically when needed (between JSON, XML and CSV), and when no common format exists the system **fails safely** with a clear, auditable log.

```
 NEGOTIATE ──► CONVERT (only if needed) ──► DELIVER
     │
     └── no common format ──► FALLBACK (abort and log)
```

**Tools:** Python standard library only (`json`, `xml.etree.ElementTree`, `csv`). No API key and no installs needed.

> **How to run:** choose **Runtime → Run all**.

## Step 1: Import the libraries

In [1]:
import csv
import io
import json
import xml.etree.ElementTree as ET
from typing import Any, Dict, List, Optional

## Step 2: Format converters (JSON, XML, CSV)

Each format needs just two functions: an **encoder** (Python dict → bytes) and a **decoder** (bytes → dict). They're stored in a small registry, so converting between *any* two formats works the same way: decode the source into a dict, then encode it into the destination. Adding a new format means writing two functions and adding one line to `FORMATS`.

XML and CSV store every value as text, so their decoders turn number-like values back into numbers. That way `101` stays an integer and `123.45` stays a float after a round trip.

In [2]:
def coerce_value(text: str) -> Any:
    """Turn number-like text back into an int or float (XML and CSV store everything as text)"""
    try:
        num = float(text)
        return int(num) if num.is_integer() else num
    except ValueError:
        return text


# --- JSON ---
def dict_to_json_bytes(d: Dict) -> bytes:
    return json.dumps(d, separators=(",", ":"), ensure_ascii=False).encode("utf-8")


def json_bytes_to_dict(b: bytes) -> Dict:
    return json.loads(b.decode("utf-8"))


# --- XML ---
def dict_to_xml_bytes(d: Dict) -> bytes:
    root = ET.Element("message")
    for k, v in d.items():
        child = ET.SubElement(root, str(k))
        child.text = str(v)
    return ET.tostring(root, encoding="utf-8")


def xml_bytes_to_dict(b: bytes) -> Dict:
    root = ET.fromstring(b.decode("utf-8"))
    return {child.tag: coerce_value(child.text or "") for child in root}


# --- CSV (new): a header row of field names, then one row of values ---
def dict_to_csv_bytes(d: Dict) -> bytes:
    buf = io.StringIO()
    writer = csv.DictWriter(buf, fieldnames=list(d.keys()), lineterminator="\n")
    writer.writeheader()
    writer.writerow(d)
    return buf.getvalue().encode("utf-8")


def csv_bytes_to_dict(b: bytes) -> Dict:
    rows = list(csv.DictReader(io.StringIO(b.decode("utf-8"))))
    if len(rows) != 1:
        raise ValueError(f"Expected exactly one CSV data row, got {len(rows)}")
    return {k: coerce_value(v) for k, v in rows[0].items()}


# --- Format registry: (encoder, decoder, content type) ---
FORMATS = {
    "JSON": (dict_to_json_bytes, json_bytes_to_dict, "application/json"),
    "XML":  (dict_to_xml_bytes,  xml_bytes_to_dict,  "application/xml"),
    "CSV":  (dict_to_csv_bytes,  csv_bytes_to_dict,  "text/csv"),
}


def get_format(fmt: str):
    try:
        return FORMATS[fmt.upper()]
    except KeyError:
        raise ValueError(f"Unsupported format: {fmt}") from None


def convert_bytes(src_fmt: str, dst_fmt: str, payload: bytes) -> bytes:
    """Convert a payload between any two registered formats via a Python dict"""
    if src_fmt.upper() == dst_fmt.upper():
        return payload
    _, decode, _ = get_format(src_fmt)
    encode, _, _ = get_format(dst_fmt)
    return encode(decode(payload))


print("Registered formats:", ", ".join(FORMATS))

Registered formats: JSON, XML, CSV


## Step 3: Negotiation

Find the formats both agents support. If there's more than one, the **sender's** preference order wins, then the receiver's. If there's no overlap, return `None`, which triggers the fallback.

In [3]:
def negotiate(sender_supported: List[str], receiver_supported: List[str],
              sender_pref: List[str], receiver_pref: List[str]) -> Optional[str]:
    s = {x.upper() for x in sender_supported}
    r = {x.upper() for x in receiver_supported}
    overlap = s & r
    if not overlap:
        return None

    # Try sender preference first, then receiver
    for p in [x.upper() for x in sender_pref]:
        if p in overlap:
            return p
    for p in [x.upper() for x in receiver_pref]:
        if p in overlap:
            return p
    return sorted(overlap)[0]  # sorted so the choice is deterministic

## Step 4: The agent model

Each `Agent` declares the formats it **supports** and its **preference** order. Encoding and the content type both come from the format registry, so the agent works with any registered format. `send()` runs the full workflow: negotiate, convert if needed, then deliver, or fall back if there's no shared format. Every step prints a log line, so each run leaves an audit trail.

In [4]:
class Agent:
    def __init__(self, name: str, supported: List[str], pref: List[str]):
        self.name = name
        self.supported = supported
        self.pref = pref

    def encode(self, fmt: str, message: Dict[str, Any]) -> bytes:
        encoder, _, _ = get_format(fmt)
        return encoder(message)

    def receive(self, payload: bytes, fmt: str) -> bool:
        return fmt.upper() in [x.upper() for x in self.supported]

    def send(self, receiver: "Agent", message: Dict[str, Any], src_format: str) -> Dict[str, Any]:
        # NEGOTIATE
        agreed = negotiate(self.supported, receiver.supported, self.pref, receiver.pref)
        print(f"NEGOTIATE: sender={self.name}, receiver={receiver.name}, "
              f"sender_supported={self.supported}, receiver_supported={receiver.supported}, "
              f"agreed={agreed}")

        if not agreed:
            # FALLBACK: no overlap, so abort safely and log it
            print("FALLBACK: No common format. Action=abort_and_log")
            return {"ok": False, "reason": "no_common_format"}

        # Prepare the "before" payload in the sender's chosen format
        payload_before = self.encode(src_format, message)

        # CONVERT only if needed
        if src_format.upper() != agreed.upper():
            print(f"CONVERT: {src_format.upper()} -> {agreed.upper()}")
            payload_after = convert_bytes(src_format, agreed, payload_before)
        else:
            print(f"CONVERT: not needed (already {agreed.upper()})")
            payload_after = payload_before

        # Show the payload before and after so the format change is visible
        try:
            print(f"PAYLOAD BEFORE ({src_format.upper()}):")
            print(payload_before.decode("utf-8").rstrip())
            print(f"PAYLOAD AFTER ({agreed.upper()}):")
            print(payload_after.decode("utf-8").rstrip())
        except UnicodeDecodeError:
            pass  # keep the demo robust even if decoding fails

        # DELIVER
        _, _, content_type = get_format(agreed)
        ok = receiver.receive(payload_after, agreed)
        print(f"DELIVER: content_type={content_type}, ok={ok}")
        return {"ok": ok, "content_type": content_type, "agreed_format": agreed}

## Step 5: The test message

The same message is sent in every scenario.

In [5]:
message = {"id": 101, "title": "Quarterly Report", "amount": 123.45}

## Step 6: Run the scenarios

### Scenario 1 — XML → JSON conversion

Sender **A** supports XML and JSON and prefers XML. Receiver **B** only understands JSON. The only shared format is JSON, so A's XML payload is converted before delivery.

In [6]:
A = Agent("A", ["XML", "JSON"], ["XML", "JSON"])  # supports both, prefers XML
B = Agent("B", ["JSON"], ["JSON"])                # JSON only

print("=== Scenario 1: XML -> JSON convert ===")
result1 = A.send(B, message, src_format="XML")
print("RESULT:", result1)

=== Scenario 1: XML -> JSON convert ===
NEGOTIATE: sender=A, receiver=B, sender_supported=['XML', 'JSON'], receiver_supported=['JSON'], agreed=JSON
CONVERT: XML -> JSON
PAYLOAD BEFORE (XML):
<message><id>101</id><title>Quarterly Report</title><amount>123.45</amount></message>
PAYLOAD AFTER (JSON):
{"id":101,"title":"Quarterly Report","amount":123.45}
DELIVER: content_type=application/json, ok=True
RESULT: {'ok': True, 'content_type': 'application/json', 'agreed_format': 'JSON'}


### Scenario 2 — JSON → XML conversion

Now sender **A** prefers JSON, and receiver **B** only understands XML. Negotiation agrees on XML, so A's JSON payload is converted.

In [7]:
A = Agent("A", ["XML", "JSON"], ["JSON", "XML"])  # supports both, prefers JSON
B = Agent("B", ["XML"], ["XML"])                  # XML only

print("=== Scenario 2: JSON -> XML convert ===")
result2 = A.send(B, message, src_format="JSON")
print("RESULT:", result2)

=== Scenario 2: JSON -> XML convert ===
NEGOTIATE: sender=A, receiver=B, sender_supported=['XML', 'JSON'], receiver_supported=['XML'], agreed=XML
CONVERT: JSON -> XML
PAYLOAD BEFORE (JSON):
{"id":101,"title":"Quarterly Report","amount":123.45}
PAYLOAD AFTER (XML):
<message><id>101</id><title>Quarterly Report</title><amount>123.45</amount></message>
DELIVER: content_type=application/xml, ok=True
RESULT: {'ok': True, 'content_type': 'application/xml', 'agreed_format': 'XML'}


### Scenario 3 — Fallback (no common format)

Agent **C** only speaks XML and agent **D** only speaks JSON. There's nothing to negotiate, so the system aborts safely and logs why instead of sending something D can't read.

In [8]:
C = Agent("C", ["XML"], ["XML"])    # XML only
D = Agent("D", ["JSON"], ["JSON"])  # JSON only

print("=== Scenario 3: Fallback (no overlap) ===")
result3 = C.send(D, message, src_format="XML")
print("RESULT:", result3)

=== Scenario 3: Fallback (no overlap) ===
NEGOTIATE: sender=C, receiver=D, sender_supported=['XML'], receiver_supported=['JSON'], agreed=None
FALLBACK: No common format. Action=abort_and_log
RESULT: {'ok': False, 'reason': 'no_common_format'}


### Scenario 4 — No conversion needed

Both agents support JSON, and the sender already sends JSON, so the payload is delivered unchanged. This shows that conversion only happens when it's actually needed.

In [9]:
E = Agent("E", ["JSON", "XML"], ["JSON"])
F = Agent("F", ["JSON", "XML"], ["XML"])

print("=== Scenario 4: Same format, no conversion ===")
result4 = E.send(F, message, src_format="JSON")
print("RESULT:", result4)

=== Scenario 4: Same format, no conversion ===
NEGOTIATE: sender=E, receiver=F, sender_supported=['JSON', 'XML'], receiver_supported=['JSON', 'XML'], agreed=JSON
CONVERT: not needed (already JSON)
PAYLOAD BEFORE (JSON):
{"id":101,"title":"Quarterly Report","amount":123.45}
PAYLOAD AFTER (JSON):
{"id":101,"title":"Quarterly Report","amount":123.45}
DELIVER: content_type=application/json, ok=True
RESULT: {'ok': True, 'content_type': 'application/json', 'agreed_format': 'JSON'}


### Scenario 5 — CSV → JSON conversion (new format)

Agent **G** is a spreadsheet-style agent: it produces CSV, but can also send JSON. Receiver **J** only understands JSON. They agree on JSON, so G's CSV payload is converted before delivery.

In [10]:
G = Agent("G", ["CSV", "JSON"], ["CSV"])   # spreadsheet agent: exports CSV, can also send JSON
J = Agent("J", ["JSON"], ["JSON"])          # JSON only

print("=== Scenario 5: CSV -> JSON convert ===")
result5 = G.send(J, message, src_format="CSV")
print("RESULT:", result5)

=== Scenario 5: CSV -> JSON convert ===
NEGOTIATE: sender=G, receiver=J, sender_supported=['CSV', 'JSON'], receiver_supported=['JSON'], agreed=JSON
CONVERT: CSV -> JSON
PAYLOAD BEFORE (CSV):
id,title,amount
101,Quarterly Report,123.45
PAYLOAD AFTER (JSON):
{"id":101,"title":"Quarterly Report","amount":123.45}
DELIVER: content_type=application/json, ok=True
RESULT: {'ok': True, 'content_type': 'application/json', 'agreed_format': 'JSON'}


### Scenario 6 — XML → CSV conversion

Sender **A** produces XML. Receiver **K** is a reporting tool that only reads CSV. Negotiation agrees on CSV and the XML payload is converted. Conversions work between any pair of registered formats, not only to and from JSON.

In [11]:
A = Agent("A", ["XML", "CSV"], ["XML", "CSV"])  # prefers XML, can also send CSV
K = Agent("K", ["CSV"], ["CSV"])                # CSV only

print("=== Scenario 6: XML -> CSV convert ===")
result6 = A.send(K, message, src_format="XML")
print("RESULT:", result6)

=== Scenario 6: XML -> CSV convert ===
NEGOTIATE: sender=A, receiver=K, sender_supported=['XML', 'CSV'], receiver_supported=['CSV'], agreed=CSV
CONVERT: XML -> CSV
PAYLOAD BEFORE (XML):
<message><id>101</id><title>Quarterly Report</title><amount>123.45</amount></message>
PAYLOAD AFTER (CSV):
id,title,amount
101,Quarterly Report,123.45
DELIVER: content_type=text/csv, ok=True
RESULT: {'ok': True, 'content_type': 'text/csv', 'agreed_format': 'CSV'}


## Step 7: Check the results

A few quick checks confirm that every scenario behaved as expected and that converting XML ↔ JSON doesn't lose or change any data.

In [12]:
# Each scenario produced the expected outcome
assert result1 == {"ok": True, "content_type": "application/json", "agreed_format": "JSON"}
assert result2 == {"ok": True, "content_type": "application/xml", "agreed_format": "XML"}
assert result3 == {"ok": False, "reason": "no_common_format"}
assert result4["agreed_format"] == "JSON" and result4["ok"]
assert result5 == {"ok": True, "content_type": "application/json", "agreed_format": "JSON"}
assert result6 == {"ok": True, "content_type": "text/csv", "agreed_format": "CSV"}

# Every pair of formats converts without losing or changing data (including types)
for src in FORMATS:
    for dst in FORMATS:
        encode_src = get_format(src)[0]
        decode_dst = get_format(dst)[1]
        back = decode_dst(convert_bytes(src, dst, encode_src(message)))
        assert back == message, (src, dst, back)
        assert type(back["id"]) is int and type(back["amount"]) is float, (src, dst)
print(f"Round trips passed for all {len(FORMATS) ** 2} format pairs")

# CSV handles values that contain commas and quotes
tricky = {"id": 7, "title": 'Report, "Q4" final', "amount": 9.5}
assert csv_bytes_to_dict(dict_to_csv_bytes(tricky)) == tricky

# Summary table
rows = [("1", "XML -> JSON", result1), ("2", "JSON -> XML", result2),
        ("3", "No overlap", result3), ("4", "Same format", result4),
        ("5", "CSV -> JSON", result5), ("6", "XML -> CSV", result6)]
print(f"{'#':<3}{'Scenario':<14}{'Delivered':<11}{'Agreed format':<15}Note")
print("-" * 62)
for n, name, r in rows:
    print(f"{n:<3}{name:<14}{str(r['ok']):<11}{r.get('agreed_format', '-'):<15}{r.get('reason', r.get('content_type'))}")
print("\nAll checks passed ✅")

Round trips passed for all 9 format pairs
#  Scenario      Delivered  Agreed format  Note
--------------------------------------------------------------
1  XML -> JSON   True       JSON           application/json
2  JSON -> XML   True       XML            application/xml
3  No overlap    False      -              no_common_format
4  Same format   True       JSON           application/json
5  CSV -> JSON   True       JSON           application/json
6  XML -> CSV    True       CSV            text/csv

All checks passed ✅


## Conclusion

The agents declared their capabilities, agreed on a common format at runtime, converted payloads automatically between JSON, XML and CSV when needed, and failed safely with a clear log when no common format existed.

**Ideas to extend it:**
- Add a fourth format, such as YAML (needs PyYAML) or TOML, by writing one encoder and one decoder and adding it to `FORMATS`
- Support nested data: the XML converter here only handles flat key–value messages
- Return a structured audit log (a list of events) instead of printing it
- Let the receiver's preference win, or score the shared formats, and compare the results